# Common setup

Run these cells first.

In [1]:
from pathlib import Path
import csv
import json
import shutil
import subprocess
import sys
from collections import Counter

RUNPOD_ROOT = Path("/workspace/SKN27-FINAL-3Team")
PROJECT_ROOT = None
if RUNPOD_ROOT.exists() and (RUNPOD_ROOT / "requirements.txt").exists():
    PROJECT_ROOT = RUNPOD_ROOT
else:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "requirements.txt").exists() and (candidate / "ai").exists() and (candidate / "storage").exists():
            PROJECT_ROOT = candidate
            break
if PROJECT_ROOT is None:
    raise FileNotFoundError("project root not found")
MANIFEST_DIR = PROJECT_ROOT / "storage/vision/datasets/classification/manifests"
RAW_VIDEO_DIR = PROJECT_ROOT / "storage/vision/datasets/classification/raw_videos"
CLIP_DIR = PROJECT_ROOT / "storage/vision/datasets/classification/clips_5s"
MODEL_DIR = PROJECT_ROOT / "storage/vision/models"
SAMPLE_MANIFEST = MANIFEST_DIR / "sample_700_coarse_manifest.csv"
DOWNLOAD_MANIFEST = MANIFEST_DIR / "train_700_download_manifest.csv"
CLIP_MANIFEST = MANIFEST_DIR / "train_700_clip_manifest_5s.csv"

PER_LABEL = 700
SEED = 42
DEVICE = "auto"
print("PROJECT_ROOT:", PROJECT_ROOT)


PROJECT_ROOT: /workspace/SKN27-FINAL-3Team


In [2]:
def run_command(command, *, timeout=None):
    command = list(map(str, command))
    print("$", " ".join(command), flush=True)
    completed = subprocess.run(command, cwd=PROJECT_ROOT, text=True, timeout=timeout)
    completed.check_returncode()
    return completed


## Install and environment check

In [3]:
run_command([sys.executable, "-m", "pip", "install", "-r", "requirements-vision-runpod.txt"], timeout=3600)
usage = shutil.disk_usage(PROJECT_ROOT)
print("free_gb:", round(usage.free / 1024**3, 2))
run_command([sys.executable, "-c", "import torch; print('cuda_available:', torch.cuda.is_available()); print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)"])


$ /usr/bin/python -m pip install -r requirements-vision-runpod.txt
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 MB 136.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.9/16.9 MB 315.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 147.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.6/676.6 kB 151.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 155.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 161.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 770.3/770.3 kB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 194.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 154.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.6/837.6 kB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 MB 145.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip


free_gb: 264087.42
$ /usr/bin/python -c import torch; print('cuda_available:', torch.cuda.is_available()); print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
cuda_available: True
device: NVIDIA RTX A5000


CompletedProcess(args=['/usr/bin/python', '-c', "import torch; print('cuda_available:', torch.cuda.is_available()); print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)"], returncode=0)

## Build or check downloaded-video manifest

In [4]:
if not SAMPLE_MANIFEST.exists():
    raise FileNotFoundError(f"sample manifest not found: {SAMPLE_MANIFEST}")

if not DOWNLOAD_MANIFEST.exists():
    run_command([
        sys.executable,
        "etl/vision/download_sampled_media.py",
        "--input", SAMPLE_MANIFEST,
        "--output", DOWNLOAD_MANIFEST,
        "--download-dir", RAW_VIDEO_DIR,
        "--label-column", "coarse_label",
        "--per-label", str(PER_LABEL),
        "--split", "",
    ], timeout=None)

with DOWNLOAD_MANIFEST.open("r", encoding="utf-8", newline="") as f:
    download_rows = list(csv.DictReader(f))
print("download_rows:", len(download_rows))
print("coarse_label_counts:", dict(Counter(row.get("coarse_label") for row in download_rows)))
print("download_status_counts:", dict(Counter(row.get("download_status") for row in download_rows)))


download_rows: 2800
coarse_label_counts: {'차대보행자': 700, '차대이륜차': 700, '차대자전거': 700, '차대차': 700}
download_status_counts: {'exists': 1370, 'downloaded': 1430}


# Model 1: YOLO/ByteTrack clip candidates + VideoMAE

Build total 5-second clips around an accident candidate, then train VideoMAE on those clips.

In [5]:
ACCIDENT_SOURCE = "yolo_track"  # use "center" if ByteTrack is too slow for all videos
YOLO_MODEL = "yolov8n.pt"
REBUILD_CLIPS = False  # True로 바꾸면 기존 clip manifest와 clip 파일을 다시 생성합니다.

if REBUILD_CLIPS or not CLIP_MANIFEST.exists():
    run_command([
        sys.executable,
        "etl/vision/build_training_clips.py",
        "--input", DOWNLOAD_MANIFEST,
        "--output", CLIP_MANIFEST,
        "--clip-dir", CLIP_DIR,
        "--label-column", "coarse_label",
        "--clip-sec", "5",
        "--short-video-sec", "5",
        "--accident-source", ACCIDENT_SOURCE,
        "--model-name", YOLO_MODEL,
        "--overwrite",
    ], timeout=None)
else:
    print("reuse existing clip manifest:", CLIP_MANIFEST)

with CLIP_MANIFEST.open("r", encoding="utf-8", newline="") as f:
    clip_rows = list(csv.DictReader(f))
clip_rows = [row for row in clip_rows if row.get("clip_status") == "ok"]
print("clip_rows_ok:", len(clip_rows))
print("clip_status_counts:", dict(Counter(row.get("clip_status") for row in clip_rows)))
print("clip_basis_counts:", dict(Counter(row.get("clip_basis") for row in clip_rows)))


reuse existing clip manifest: /workspace/SKN27-FINAL-3Team/storage/vision/datasets/classification/manifests/train_700_clip_manifest_5s.csv
clip_rows_ok: 2797
clip_status_counts: {'ok': 2797}
clip_basis_counts: {'yolo_bytetrack_bbox_change': 2797}


## Shared VideoMAE helpers

In [6]:
VIDEOMAE_MODEL_DIR = MODEL_DIR / "videomae_classification_clip5s"
EARLY_STOPPING_PATIENCE = 2

def build_videomae_command(experiment):
    manifest = experiment.get("manifest", CLIP_MANIFEST)
    output_dir = experiment.get("output_dir", VIDEOMAE_MODEL_DIR)
    command = [
        sys.executable,
        "ai/vision/train_videomae_classifier.py",
        "--manifest", manifest,
        "--root-dir", PROJECT_ROOT,
        "--output-dir", output_dir,
        "--label-column", "coarse_label",
        "--frame-count", str(experiment["frame_count"]),
        "--epochs", str(experiment["epochs"]),
        "--batch-size", str(experiment["batch_size"]),
        "--learning-rate", str(experiment["learning_rate"]),
        "--weight-decay", str(experiment["weight_decay"]),
        "--early-stopping-patience", str(EARLY_STOPPING_PATIENCE),
        "--seed", str(SEED),
        "--device", DEVICE,
        "--num-workers", "0",
        "--no-show-progress",
    ]
    if experiment["freeze_backbone"]:
        command.append("--freeze-backbone")
    return command

def latest_run_dir(output_dir):
    runs = [path for path in output_dir.iterdir() if path.is_dir()]
    if not runs:
        raise FileNotFoundError(f"No run directories found: {output_dir}")
    return max(runs, key=lambda path: path.stat().st_mtime)

def show_run_result(run_dir):
    print("run_dir:", run_dir)
    for name in ["run_config.json", "training_history.csv"]:
        path = run_dir / name
        print("##", name, path.exists())
        if path.suffix == ".json" and path.exists():
            data = json.loads(path.read_text(encoding="utf-8"))
            for key in ["run_id", "freeze_backbone", "epochs", "batch_size", "learning_rate", "weight_decay", "best_epoch", "best_val_accuracy", "train_rows", "val_rows", "test_rows"]:
                if key in data:
                    print(key, data[key])
        elif path.exists():
            rows = list(csv.DictReader(path.open("r", encoding="utf-8")))
            for row in rows:
                print(row)
            if rows:
                print("best_val:", max(rows, key=lambda row: float(row.get("val_accuracy") or 0)))
                print("best_test:", max(rows, key=lambda row: float(row.get("test_accuracy") or 0)))


## Combination 1 - freeze baseline - define

In [7]:
EXPERIMENT = {'name': 'videomae_clip5s_baseline_freeze_lr1e-3_e5', 'epochs': 5, 'batch_size': 2, 'learning_rate': 0.001, 'weight_decay': 0.0, 'frame_count': 16, 'freeze_backbone': True}
print(EXPERIMENT)


{'name': 'videomae_clip5s_baseline_freeze_lr1e-3_e5', 'epochs': 5, 'batch_size': 2, 'learning_rate': 0.001, 'weight_decay': 0.0, 'frame_count': 16, 'freeze_backbone': True}


## Combination 1 - freeze baseline - train

In [8]:
run_command(build_videomae_command(EXPERIMENT), timeout=None)
LAST_RUN_DIR = latest_run_dir(VIDEOMAE_MODEL_DIR)
print("LAST_RUN_DIR:", LAST_RUN_DIR)


$ /usr/bin/python ai/vision/train_videomae_classifier.py --manifest /workspace/SKN27-FINAL-3Team/storage/vision/datasets/classification/manifests/train_700_clip_manifest_5s.csv --root-dir /workspace/SKN27-FINAL-3Team --output-dir /workspace/SKN27-FINAL-3Team/storage/vision/models/videomae_classification_clip5s --label-column coarse_label --frame-count 16 --epochs 5 --batch-size 2 --learning-rate 0.001 --weight-decay 0.0 --early-stopping-patience 2 --seed 42 --device auto --num-workers 0 --no-show-progress --freeze-backbone


[transformers] You passed `num_labels=4` which is incompatible to the `id2label` map of length `400`.
Loading weights: 100%|██████████| 162/162 [00:00<00:00, 6209.41it/s]
[transformers] VideoMAEForVideoClassification LOAD REPORT from: MCG-NJU/videomae-base-finetuned-kinetics
Key                                                            | Status     |                                                                                         
---------------------------------------------------------------+------------+-----------------------------------------------------------------------------------------
videomae.encoder.layer.{0...11}.attention.attention.q_bias     | UNEXPECTED |                                                                                         
videomae.encoder.layer.{0...11}.attention.attention.v_bias     | UNEXPECTED |                                                                                         
videomae.encoder.layer.{0...11}.attention.attention.quer

run_id: videomae_cls_20260710_022831
device: cuda
manifest: /workspace/SKN27-FINAL-3Team/storage/vision/datasets/classification/manifests/train_700_clip_manifest_5s.csv
labels: {'차대보행자': 0, '차대이륜차': 1, '차대자전거': 2, '차대차': 3}
rows: train=1954 val=560 test=283
epoch=1 train_loss=1.258350 train_acc=0.433470 val_loss=1.175454 val_acc=0.460714
epoch=2 train_loss=1.097565 train_acc=0.515353 val_loss=1.092254 val_acc=0.548214
epoch=3 train_loss=1.043230 train_acc=0.558342 val_loss=1.103989 val_acc=0.514286
epoch=4 train_loss=1.003075 train_acc=0.591095 val_loss=1.124640 val_acc=0.544643
early_stopping: epoch=4 best_epoch=2 best_val_acc=0.548214


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.14it/s]


output_dir: /workspace/SKN27-FINAL-3Team/storage/vision/models/videomae_classification_clip5s/videomae_cls_20260710_022831
LAST_RUN_DIR: /workspace/SKN27-FINAL-3Team/storage/vision/models/videomae_classification_clip5s/videomae_cls_20260710_022831


## Combination 1 - freeze baseline - result

In [9]:
show_run_result(LAST_RUN_DIR)


run_dir: /workspace/SKN27-FINAL-3Team/storage/vision/models/videomae_classification_clip5s/videomae_cls_20260710_022831
## run_config.json True
run_id videomae_cls_20260710_022831
freeze_backbone True
epochs 5
batch_size 2
learning_rate 0.001
weight_decay 0.0
best_epoch 2
best_val_accuracy 0.5482142857142858
train_rows 1954
val_rows 560
test_rows 283
## training_history.csv True
{'epoch': '1', 'train_loss': '1.258350', 'train_accuracy': '0.433470', 'val_loss': '1.175454', 'val_accuracy': '0.460714', 'test_loss': '1.237618', 'test_accuracy': '0.469965'}
{'epoch': '2', 'train_loss': '1.097565', 'train_accuracy': '0.515353', 'val_loss': '1.092254', 'val_accuracy': '0.548214', 'test_loss': '1.184257', 'test_accuracy': '0.530035'}
{'epoch': '3', 'train_loss': '1.043230', 'train_accuracy': '0.558342', 'val_loss': '1.103989', 'val_accuracy': '0.514286', 'test_loss': '1.170791', 'test_accuracy': '0.498233'}
{'epoch': '4', 'train_loss': '1.003075', 'train_accuracy': '0.591095', 'val_loss': '1.1

## Combination 2 - unfreeze lr 1e-4 - define

In [10]:
EXPERIMENT = {'name': 'videomae_clip5s_exp2_lr1e-4_e10', 'epochs': 10, 'batch_size': 2, 'learning_rate': 0.0001, 'weight_decay': 0.0, 'frame_count': 16, 'freeze_backbone': False}
print(EXPERIMENT)


{'name': 'videomae_clip5s_exp2_lr1e-4_e10', 'epochs': 10, 'batch_size': 2, 'learning_rate': 0.0001, 'weight_decay': 0.0, 'frame_count': 16, 'freeze_backbone': False}


## Combination 2 - unfreeze lr 1e-4 - train

In [ ]:
run_command(build_videomae_command(EXPERIMENT), timeout=None)
LAST_RUN_DIR = latest_run_dir(VIDEOMAE_MODEL_DIR)
print("LAST_RUN_DIR:", LAST_RUN_DIR)


$ /usr/bin/python ai/vision/train_videomae_classifier.py --manifest /workspace/SKN27-FINAL-3Team/storage/vision/datasets/classification/manifests/train_700_clip_manifest_5s.csv --root-dir /workspace/SKN27-FINAL-3Team --output-dir /workspace/SKN27-FINAL-3Team/storage/vision/models/videomae_classification_clip5s --label-column coarse_label --frame-count 16 --epochs 10 --batch-size 2 --learning-rate 0.0001 --weight-decay 0.0 --early-stopping-patience 2 --seed 42 --device auto --num-workers 0 --no-show-progress


[transformers] You passed `num_labels=4` which is incompatible to the `id2label` map of length `400`.
Loading weights: 100%|██████████| 162/162 [00:00<00:00, 8910.36it/s]
[transformers] VideoMAEForVideoClassification LOAD REPORT from: MCG-NJU/videomae-base-finetuned-kinetics
Key                                                            | Status     |                                                                                         
---------------------------------------------------------------+------------+-----------------------------------------------------------------------------------------
videomae.encoder.layer.{0...11}.attention.attention.q_bias     | UNEXPECTED |                                                                                         
videomae.encoder.layer.{0...11}.attention.attention.v_bias     | UNEXPECTED |                                                                                         
videomae.encoder.layer.{0...11}.attention.attention.quer

## Combination 2 - unfreeze lr 1e-4 - result

In [ ]:
show_run_result(LAST_RUN_DIR)


## Combination 3 - regularized lr 5e-5 - define

In [ ]:
EXPERIMENT = {'name': 'videomae_clip5s_exp3_lr5e-5_wd5e-2_e30', 'epochs': 30, 'batch_size': 2, 'learning_rate': 5e-05, 'weight_decay': 0.05, 'frame_count': 16, 'freeze_backbone': False}
print(EXPERIMENT)


## Combination 3 - regularized lr 5e-5 - train

In [ ]:
run_command(build_videomae_command(EXPERIMENT), timeout=None)
LAST_RUN_DIR = latest_run_dir(VIDEOMAE_MODEL_DIR)
print("LAST_RUN_DIR:", LAST_RUN_DIR)


## Combination 3 - regularized lr 5e-5 - result

In [ ]:
show_run_result(LAST_RUN_DIR)


## 결과 검토 및 다음 실험

현재까지의 VideoMAE 기준 최고 결과는 조합 3입니다. 설정은 `lr=5e-5`, `weight_decay=0.05`, backbone unfreeze이며, 검증 정확도는 약 `0.552`, 테스트 정확도는 약 `0.604`입니다. 이후에는 train 정확도만 오르고 validation/test 성능은 크게 오르지 않으므로 epoch를 무작정 늘리기보다 과적합을 줄이는 실험이 우선입니다.

또한 5초 clip manifest를 먼저 확인해야 합니다. 특정 라벨의 clip 수가 다른 라벨보다 크게 줄어들면, 모델 파라미터를 조정해도 결과가 안정적으로 개선되기 어렵습니다.


In [ ]:
from collections import Counter

for manifest_path in [DOWNLOAD_MANIFEST, CLIP_MANIFEST]:
    print(chr(10) + "##", manifest_path.name)
    with manifest_path.open("r", encoding="utf-8", newline="") as f:
        rows = list(csv.DictReader(f))
    print("rows:", len(rows))
    for column in ["coarse_label", "split", "clip_status", "file_exists"]:
        if rows and column in rows[0]:
            print(column, dict(Counter(row.get(column) for row in rows)))


## Combination 4 - lower lr with same regularization

Keep the best direction from Combination 3, but lower the learning rate to reduce overfitting after epoch 5.


In [ ]:
EXPERIMENT = {'name': 'videomae_clip5s_exp4_lr3e-5_wd5e-2_e20', 'epochs': 20, 'batch_size': 2, 'learning_rate': 3e-05, 'weight_decay': 0.05, 'frame_count': 16, 'freeze_backbone': False}
print(EXPERIMENT)


## Combination 4 - train

In [ ]:
run_command(build_videomae_command(EXPERIMENT), timeout=None)
LAST_RUN_DIR = latest_run_dir(VIDEOMAE_MODEL_DIR)
print("LAST_RUN_DIR:", LAST_RUN_DIR)


## Combination 4 - result

In [ ]:
show_run_result(LAST_RUN_DIR)


## Combination 5 - stronger regularization

Use the same learning rate as the current best run, but increase weight decay. This tests whether the overfitting after epoch 5 is mainly regularization-related.


In [ ]:
EXPERIMENT = {'name': 'videomae_clip5s_exp5_lr5e-5_wd1e-1_e20', 'epochs': 20, 'batch_size': 2, 'learning_rate': 5e-05, 'weight_decay': 0.1, 'frame_count': 16, 'freeze_backbone': False}
print(EXPERIMENT)


## Combination 5 - train

In [ ]:
run_command(build_videomae_command(EXPERIMENT), timeout=None)
LAST_RUN_DIR = latest_run_dir(VIDEOMAE_MODEL_DIR)
print("LAST_RUN_DIR:", LAST_RUN_DIR)


## Combination 5 - result

In [ ]:
show_run_result(LAST_RUN_DIR)


## BBox/사고 후보 중심 crop 실험

YOLO/ByteTrack으로 잡은 사고 후보 객체쌍의 bbox union 주변을 crop한 5초 영상을 별도로 만들고, 같은 VideoMAE 설정으로 비교합니다. 기존 full-frame clip 실험은 그대로 둡니다.


In [ ]:
BBOX_CLIP_DIR = PROJECT_ROOT / "storage/vision/datasets/classification/clips_5s_bbox_crop"
BBOX_CLIP_MANIFEST = MANIFEST_DIR / "train_700_clip_manifest_5s_bbox_crop.csv"
BBOX_VIDEOMAE_MODEL_DIR = MODEL_DIR / "videomae_classification_clip5s_bbox_crop"
REBUILD_BBOX_CROP_CLIPS = False

if REBUILD_BBOX_CROP_CLIPS or not BBOX_CLIP_MANIFEST.exists():
    run_command([
        sys.executable,
        "etl/vision/build_training_clips.py",
        "--input", DOWNLOAD_MANIFEST,
        "--output", BBOX_CLIP_MANIFEST,
        "--clip-dir", BBOX_CLIP_DIR,
        "--label-column", "coarse_label",
        "--clip-sec", "5",
        "--short-video-sec", "5",
        "--accident-source", ACCIDENT_SOURCE,
        "--model-name", YOLO_MODEL,
        "--crop-mode", "bbox",
        "--crop-padding-ratio", "0.35",
        "--overwrite",
    ], timeout=None)
else:
    print("reuse existing bbox crop manifest:", BBOX_CLIP_MANIFEST)

with BBOX_CLIP_MANIFEST.open("r", encoding="utf-8", newline="") as f:
    bbox_crop_rows = [row for row in csv.DictReader(f) if row.get("clip_status") == "ok"]
print("bbox_crop_rows_ok:", len(bbox_crop_rows))
print("bbox_crop_label_counts:", dict(Counter(row.get("coarse_label") for row in bbox_crop_rows)))
print("bbox_crop_basis_counts:", dict(Counter(row.get("clip_basis") for row in bbox_crop_rows)))


## BBox crop Combination 1 - define


In [ ]:
EXPERIMENT = {
    "name": "videomae_bbox_crop_exp1_freeze_lr1e-4_e10",
    "manifest": BBOX_CLIP_MANIFEST,
    "output_dir": BBOX_VIDEOMAE_MODEL_DIR,
    "epochs": 10,
    "batch_size": 2,
    "learning_rate": 0.0001,
    "weight_decay": 0.05,
    "frame_count": 16,
    "freeze_backbone": True,
}
print(EXPERIMENT)


## BBox crop Combination 1 - train


In [ ]:
run_command(build_videomae_command(EXPERIMENT), timeout=None)
LAST_RUN_DIR = latest_run_dir(BBOX_VIDEOMAE_MODEL_DIR)
print("LAST_RUN_DIR:", LAST_RUN_DIR)


## BBox crop Combination 1 - result


In [ ]:
show_run_result(LAST_RUN_DIR)
